In [ ]:
%pip install geopandas shapely pyproj rasterio contextily folium scikit-learn jupyterlab xgboost

In [ ]:
%pip list

In [ ]:
# import requests

# url    = "https://phl.carto.com/api/v2/sql"
# params = {
#     "q": """
#       SELECT
#         *,
#         ST_AsGeoJSON(the_geom)::json AS geometry
#       FROM assessments
#       WHERE year > 2022
#     """,
#     "format": "GeoJSON"
# }

# resp = requests.get(url, params=params)
# resp.raise_for_status()

# with open("data/opa_properties_public.geojson", "w") as f:
#     f.write(resp.text)

import requests

url    = "https://phl.carto.com/api/v2/sql"
params = {
    "q": """
      SELECT
        *,
        ST_AsGeoJSON(the_geom)::json AS geometry
      FROM opa_properties_public
      WHERE assessment_date >= '2022-01-01'
    """,
    "format": "GeoJSON"
}

resp = requests.get(url, params=params)
resp.raise_for_status()

props = resp.json()
print(f"Retrieved {len(resp['features'])} features.")
with open("data/opa_properties_public.geojson", "w") as f:
    f.write(props.text)
    

In [ ]:
import requests

# Endpoint and parameters
url = "https://phl.carto.com/api/v2/sql"
params = {
    "q": "SELECT *, ST_AsGeoJSON(the_geom)::json AS the_geom_geojson FROM public_cases_fc WHERE requested_datetime >= '2022-01-01'",
    "format": "GeoJSON"
}

# Send the GET request
resp = requests.get(url, params=params)
resp.raise_for_status()  # will raise if response code is 4xx/5xx

# Parse as GeoJSON
calls = resp.json()
print(f"Retrieved {len(calls['features'])} features.")
with open("data/311.geojson", "w") as f:
    f.write(resp.text)


In [ ]:
import geopandas as gpd
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
import folium
import numpy as np

In [ ]:
# 1. Load property polygons

props = gpd.read_file("data/opa_properties_public.geojson").to_crs(3857)

# 2. Load 311 point data and spatial‑join to parcels

calls = gpd.read_file("data/311.geojson").to_crs(3857)

joined = gpd.sjoin(props, calls[['geometry','service_name']], how="left")

In [ ]:
props

In [ ]:
calls

In [ ]:
joined

In [ ]:
# 3. Feature engineering: count complaints per parcel
feat = (joined.groupby("parcel_number").service_name.count().rename("complaint_ct").reset_index())

df = props.merge(feat, on="parcel_number", how="left")

df["complaint_ct"] = df["complaint_ct"].fillna(0).astype(int)

df

In [ ]:
df[df['complaint_ct'] > 0]

In [ ]:
df["shape_area"] = props.geometry.area

In [ ]:
df_cleaned = df.drop(index=df[df["sale_price"].isnull()].index)

In [ ]:
import xgboost as xgb

# 4. Simple model: predict log(sale_price) from complaint_ct & area
X = df_cleaned[["complaint_ct", "shape_area"]]

y = np.log1p(df_cleaned["sale_price"])

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=.2, random_state=42)

# m = RandomForestRegressor(n_estimators=200).fit(X_train, y_train)

# print("Test R²:", m.score(X_test, y_test))

model = xgb.XGBRegressor(
    n_estimators=1000,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.7,
    colsample_bytree=0.8,
    random_state=42
)

model.fit(X_train, y_train,
          eval_set=[(X_train, y_train),(X_test, y_test)],
          early_stopping_rounds=50,
          verbose=False)

print("XGB R²:", model.score(X_test, y_test))

In [ ]:
# 5. Visualise predictions on an interactive map

df_cleaned["pred_price"] = np.expm1(m.predict(X))

# reproject for mapping
gd = df_cleaned.to_crs(4326)

# pull out only the fields Folium needs:
geojson = {
    "type": "FeatureCollection",
    "features": []
}
for _, row in gd.iterrows():
    geom = row.geometry
    if geom is None:
        continue   # skip parcels with no geometry
    geojson["features"].append({
        "type": "Feature",
        "geometry": geom.__geo_interface__,
        "properties": {
            "parcel_number": row.parcel_number,
            "pred_price": row.pred_price
        }
    })

In [ ]:
import json
with open("data/philly_parcels.geojson", "w") as f:
    json.dump(geojson, f)

In [ ]:
df_cleaned.to_csv("data/philly_parcels.csv", index=False)

In [ ]:
# Basic but does not scale well 
mmap = folium.Map(location=[39.95,-75.17], zoom_start=11, tiles="cartodbpositron")

folium.Choropleth(geo_data=geojson,
                  data=df_cleaned,
                  columns=["parcel_number","pred_price"],
                  key_on="feature.properties.parcel_number",
                  fill_color="YlGnBu",
                  legend_name="Predicted Sale Price").add_to(mmap)

mmap.save("philly_parcels.html")

# Scaled Solution

In [ ]:
%%bash
tippecanoe --force -o parcels.mbtiles -Z 10 -z 16 --drop-densest-as-needed data/philly_parcels.geojson

# install a lightweight MBTiles tile server
npm install -g tileserver-gl-light

# point it at your tileset
tileserver-gl-light parcels.mbtiles

In [ ]:
# from IPython.display import IFrame
# IFrame("philly_parcels.html", width=800, height=600)

In [ ]:
%%bash
python -m http.server 8000